# K-Means Clustering
**INDE 577 / CMOR 438 — Jingru Wu, Rice University**

---

## 1. Mathematical Background

K-Means partitions $n$ samples into $K$ clusters by minimising the **within-cluster sum of squares (WCSS)** — also called *inertia*:

$$\mathcal{J} = \sum_{k=1}^K \sum_{\mathbf{x}_i \in C_k} \| \mathbf{x}_i - \boldsymbol{\mu}_k \|_2^2$$

where $\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{\mathbf{x}_i \in C_k} \mathbf{x}_i$ is the centroid of cluster $k$.

### Lloyd's Algorithm
1. **Initialise** $K$ centroids (k-means++ for better spread)
2. **Assign** each point to the nearest centroid: $\hat{k} = \arg\min_k \|\mathbf{x} - \boldsymbol{\mu}_k\|^2$
3. **Update** centroids as the mean of assigned points
4. **Repeat** until convergence (centroid shift $< \epsilon$)

### k-means++ Initialisation
Seeds centroids proportional to their squared distance from existing centroids, reducing sensitivity to random initialisation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans as SklearnKMeans

import sys
sys.path.insert(0, '../../src')
from rice_ml.unsupervised_learning.kmeans import KMeans
from rice_ml.unsupervised_learning.pca import PCA

print('rice_ml imported successfully')

## 2. Synthetic Data — Three Isotropic Blobs

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='tab10', s=20, alpha=0.8)
plt.title('Ground-Truth Clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.tight_layout()
plt.show()

## 3. Training & Visualisation

In [ ]:
model = KMeans(n_clusters=3, random_state=42)
labels = model.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, lbl, title in zip(axes, [y_true, labels], ['Ground Truth', 'K-Means (rice_ml)']):
    ax.scatter(X[:, 0], X[:, 1], c=lbl, cmap='tab10', s=20, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Plot centroids
axes[1].scatter(model.centroids[:, 0], model.centroids[:, 1],
                marker='X', s=200, c='red', zorder=5, label='Centroids')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f'Inertia : {model.inertia_:.2f}')
print(f'Converged in {model.n_iter_} iterations')

## 4. Elbow Method — Choosing K

In [ ]:
inertias = []
K_range = range(1, 10)

for k in K_range:
    m = KMeans(n_clusters=k, random_state=0).fit(X)
    inertias.append(m.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(K_range), inertias, 'bo-')
plt.axvline(3, color='red', linestyle='--', label='True K=3')
plt.xlabel('Number of Clusters K')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow Method')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Real Data — Iris Dataset with PCA Visualisation

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
Xi, yi = iris.data, iris.target

# Reduce to 2D for visualisation
pca = PCA(n_components=2)
Xi_2d = pca.fit_transform(Xi)

km_iris = KMeans(n_clusters=3, random_state=42).fit(Xi)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, lbl, title in zip(axes, [yi, km_iris.labels_], ['Ground Truth (Iris)', 'K-Means Labels']):
    ax.scatter(Xi_2d[:, 0], Xi_2d[:, 1], c=lbl, cmap='tab10', s=25, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
plt.tight_layout()
plt.show()

print(f'Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.2%}')

## 6. Comparison with scikit-learn

In [ ]:
sk_km = SklearnKMeans(n_clusters=3, random_state=42, n_init=10).fit(X)

print(f'rice_ml inertia : {model.inertia_:.4f}')
print(f'sklearn inertia : {sk_km.inertia_:.4f}')

## 7. Limitations

- Assumes **spherical, equally-sized** clusters — fails on non-convex shapes (see make_moons below)
- Requires **specifying K** in advance
- Sensitive to **outliers** — consider robust alternatives like DBSCAN for noisy data
- May converge to **local minima** — k-means++ greatly reduces this risk

In [ ]:
# Failure case: non-convex shapes
X_moons, _ = make_moons(n_samples=200, noise=0.05, random_state=0)
labels_moons = KMeans(n_clusters=2, random_state=0).fit_predict(X_moons)

plt.figure(figsize=(6, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels_moons, cmap='tab10', s=20)
plt.title('K-Means on Two Moons (Failure Case)')
plt.tight_layout()
plt.show()